# PCD suite on Colab

This notebook continues the experiment queue where the AUS workstation stopped. It reads and writes the shared Google Drive folder `pcd-suite`, so its results go next to the workstation's.

**Before you run it:**
1. **Use a GPU runtime:** *Runtime → Change runtime type → T4 GPU*. Use L4 if the T4s are full.
2. **Make the Drive folder reachable.** `pcd-suite` must be shared with this Google account as **Editor**. If it was shared from another account, open *Shared with me* in Google Drive, right-click `pcd-suite`, and choose *Organize → Add shortcut → My Drive*.

Run the cells in order. The last cell keeps running: it trains and copies results to Drive every 5 minutes. If the session disconnects, choose *Runtime → Run all* again. Finished runs are skipped, and unfinished ones resume from their last checkpoint.

In [ ]:
!nvidia-smi -L
!nproc

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE = "/content/drive/MyDrive/pcd-suite"  # the folder or shortcut name in My Drive (case matters)
if not os.path.isdir(DRIVE):
    print("Not found. My Drive contains:", sorted(os.listdir("/content/drive/MyDrive")))
    raise SystemExit("Set DRIVE above to the pcd-suite folder or shortcut name, then rerun this cell.")
print(sorted(os.listdir(DRIVE)))  # expect ['logs', 'outputs', 'snapshot']

In [ ]:
# Get the code (public repository)
import subprocess
BRANCH = "Battikhah/colab"
CODE = "/content/pcd-code"
URL = "https://github.com/Battikhah/MLR570-project"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--branch", BRANCH, URL, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only", "origin", BRANCH], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Settings
os.environ.update({
    "DRIVE": DRIVE,
    "PROCS": "3",          # runs sharing the GPU at once (4 on an L4)
    "WORKERS": "1",        # data-loading workers per run: 1 if nproc printed 2-4, 2 if 8 or more
    "SYNC_MIN": "5",       # minutes between copies to Drive
    # Which experiments. With families set, the start is quick: only the teacher, small markers for
    # finished runs and the checkpoints of unfinished ones are read from Drive.
    # Empty ("") = the whole remaining queue, which reads the 1.2 GB snapshot first.
    "FAMILY": "teacher-switch teacher-corruption",
    "SKIP_TESTS": "0",     # "1" after the first session saves about 6 minutes
    "SETUP_ONLY": "0",     # "1" = prepare everything but start no runs
})

In [ ]:
# Restore from Drive, install, test, then run the queue. Keep this cell running.
!bash /content/pcd-code/scripts/colab/run_colab.sh